# Simple Groq API Test

Just calls Groq API and prints the raw output to see what we're getting.

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load API key
load_dotenv()
api_key = os.environ.get("GROQ_API_KEY")

if not api_key:
    api_key = input("Enter GROQ_API_KEY: ").strip()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key,
)

print("API client initialized")

API client initialized


In [ ]:
# Test 2: Rigorous parse_probability testing
import re

print("TEST 2: Rigorous parse_probability testing")
print("="*70)

def parse_probability(text):
    """Extract probability from response"""
    if not text or not text.strip():
        raise ValueError("Empty response")
    
    # Try to match decimal probability
    match = re.search(r'([01]?\.\d+|[01])', text.strip())
    if not match:
        raise ValueError(f"No probability in: '{text}'")
    
    p = float(match.group(1))
    if p > 1:
        p = p / 100
    if not (0 <= p <= 1):
        raise ValueError(f"Out of range: {p}")
    
    return p

# Test cases for parse_probability
test_cases = [
    ("0.23", 0.23, "Normal decimal"),
    ("0.5", 0.5, "Single decimal place"),
    ("0.001", 0.001, "Small probability"),
    ("0.999", 0.999, "Large probability"),
    ("0.0", 0.0, "Exactly zero"),
    ("1.0", 1.0, "Exactly one"),
    ("0", 0.0, "Integer zero"),
    ("1", 1.0, "Integer one"),
    ("   0.45   ", 0.45, "Whitespace padding"),
    ("The probability is 0.67", 0.67, "Embedded in text"),
    (".5", 0.5, "Missing leading zero"),
]

print("Testing parse_probability function:")
print()
all_passed = True

for input_text, expected, description in test_cases:
    try:
        result = parse_probability(input_text)
        if abs(result - expected) < 0.0001:  # Float comparison with tolerance
            status = "✓ PASS"
        else:
            status = f"✗ FAIL (got {result}, expected {expected})"
            all_passed = False
        print(f"{status:15s} | {description:25s} | Input: '{input_text:20s}' → {result}")
    except Exception as e:
        status = f"✗ ERROR: {e}"
        all_passed = False
        print(f"{status:15s} | {description:25s} | Input: '{input_text}'")

print()
if all_passed:
    print("✓ All parse tests PASSED!")
else:
    print("✗ Some tests FAILED - there's a bug in parse_probability")
print()

# Now test with the actual API response
print("Parsing actual API response:")
try:
    parsed = parse_probability(raw_output)
    print(f"Raw output: '{raw_output}'")
    print(f"Parsed probability: {parsed}")
    print(f"Type: {type(parsed)}")
    
    if parsed == 0.0:
        print()
        print("⚠️  Result is 0.0 - checking if this is correct...")
        print(f"   Raw string repr: {repr(raw_output)}")
        print(f"   Is it literally '0' or '0.0'? {raw_output.strip() in ['0', '0.0']}")
except Exception as e:
    print(f"ERROR parsing: {e}")
print()

In [ ]:
# Test 2: Parse the response
import re

print("TEST 2: Parsing the response")
print("="*70)

def parse_probability(text):
    """Extract probability from response"""
    if not text or not text.strip():
        raise ValueError("Empty response")
    
    # Try to match decimal probability
    match = re.search(r'([01]?\.\d+|[01])', text.strip())
    if not match:
        raise ValueError(f"No probability in: '{text}'")
    
    p = float(match.group(1))
    if p > 1:
        p = p / 100
    if not (0 <= p <= 1):
        raise ValueError(f"Out of range: {p}")
    
    return p

try:
    parsed = parse_probability(raw_output)
    print(f"Parsed probability: {parsed}")
    print(f"Parsed type: {type(parsed)}")
except Exception as e:
    print(f"ERROR parsing: {e}")
print()

In [3]:
# Test 3: Multiple calls with different prompts
print("TEST 3: Multiple different questions")
print("="*70)

test_questions = [
    ("Will Bitcoin hit $100k by end of 2026?", 0.45),
    ("Will it snow in Miami next week?", 0.01),
    ("Will the sun rise tomorrow?", 0.99),
    ("Will a coin flip land heads?", 0.50),
]

for question, prior in test_questions:
    user_prompt = f"""You are forecasting the probability this market resolves YES.

Question: {question}
mid_yes (prior): {prior:.2f}

Output only the final decimal probability."""
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0,
    )
    
    raw = response.choices[0].message.content
    
    try:
        parsed = parse_probability(raw)
        print(f"Q: {question[:50]:50s} | Prior: {prior:.2f} | Raw: '{raw:10s}' | Parsed: {parsed:.3f}")
    except Exception as e:
        print(f"Q: {question[:50]:50s} | Prior: {prior:.2f} | Raw: '{raw:10s}' | ERROR: {e}")

print("\nAll tests complete!")

TEST 3: Multiple different questions
Q: Will Bitcoin hit $100k by end of 2026?             | Prior: 0.45 | Raw: '0.1       ' | ERROR: name 'parse_probability' is not defined
Q: Will it snow in Miami next week?                   | Prior: 0.01 | Raw: '0.005     ' | ERROR: name 'parse_probability' is not defined
Q: Will the sun rise tomorrow?                        | Prior: 0.99 | Raw: '0.999     ' | ERROR: name 'parse_probability' is not defined
Q: Will a coin flip land heads?                       | Prior: 0.50 | Raw: '0.5       ' | ERROR: name 'parse_probability' is not defined

All tests complete!


In [ ]:
# Test 4: Check with actual market data
import pandas as pd

print("TEST 4: Test with real market data")
print("="*70)

# Load one market
df = pd.read_csv("data/markets_microstructure_v2_v3_merged.csv")
market = df.iloc[0]

print(f"Market: {market['event_title']}")
print(f"Prior (mid_yes): {market['mid_yes']:.3f}")
print()

user_prompt = f"""You are forecasting the probability this market resolves YES.

Question: {market['event_title']}
mid_yes (prior): {market['mid_yes']:.2f}

Available signals: None (baseline - only question and prior).
Use your reasoning about the question to update the prior.
Output only the final decimal probability."""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0,
)

raw = response.choices[0].message.content
parsed = parse_probability(raw)

print(f"Raw API response: '{raw}'")
print(f"Parsed probability: {parsed:.6f}")
print()

if parsed == 0.0:
    print("⚠️  WARNING: Parsed probability is exactly 0.0")
    print("   The LLM might be returning '0' or '0.0' as its actual prediction.")
    print("   This could mean the LLM thinks the probability is extremely low.")
elif parsed == market['mid_yes']:
    print("ℹ️  The parsed probability equals the prior (no update).")
else:
    print(f"✓  Prediction differs from prior: {market['mid_yes']:.3f} → {parsed:.3f}")